# Audio Capture -> Speech To Text (Whisper)\n\nThis notebook is for testing the workflow after your frontend captures and saves user audio into:\n\n- `AI_BACKEND/audio_inputs/`\n\nIt will:\n1. List captured audio files\n2. (Optional) Convert them to WAV using `ffmpeg`\n3. Transcribe with OpenAI Whisper\n4. Save transcripts to `AI_BACKEND/audio_inputs/transcripts.jsonl`

In [3]:
from __future__ import annotations
    
import os
import json
from pathlib import Path
    
AUDIO_DIR = Path("./audio_inputs").resolve()  # relative to AI_BACKEND
TRANSCRIPT_OUT = AUDIO_DIR / "transcripts.jsonl"
    
print("AUDIO_DIR:", AUDIO_DIR)
print("TRANSCRIPT_OUT:", TRANSCRIPT_OUT)
   
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
    

AUDIO_DIR: C:\Users\DELL\Desktop\AI-POWERED-INTERVIEW-PREPARATION-COACH\AI_BACKEND\audio_capture\audio_inputs
TRANSCRIPT_OUT: C:\Users\DELL\Desktop\AI-POWERED-INTERVIEW-PREPARATION-COACH\AI_BACKEND\audio_capture\audio_inputs\transcripts.jsonl


In [4]:
# 1) List captured audio files
exts = {".wav", ".mp3", ".m4a", ".mp4", ".webm", ".ogg", ".flac"}
files = [p for p in AUDIO_DIR.iterdir() if p.is_file() and p.suffix.lower() in exts]
files_sorted = sorted(files, key=lambda p: p.stat().st_mtime, reverse=True)
    
print(f"Found {len(files_sorted)} audio file(s) in audio_inputs/")
for i, p in enumerate(files_sorted[:15], start=1):
        print(i, p.name)
 

Found 0 audio file(s) in audio_inputs/


## 2) Install dependencies (only if needed)
+
+Run this cell if Whisper is not installed.
+

In [6]:
# If you need to install Whisper in your current env, uncomment:    # If you get ffmpeg errors, install ffmpeg (system-wide) and restart kernel.    # Quick import test
try:
        import whisper  # type: ignore
        print("whisper import OK")
except Exception as e:
        print("whisper import failed:", e)
  

whisper import failed: No module named 'whisper'


In [7]:
# Helper: convert to WAV if not already WAV.
import subprocess

def ensure_wav(input_path: Path) -> Path:
        if input_path.suffix.lower() == ".wav":
            return input_path

        wav_path = input_path.with_suffix(".wav")
        if wav_path.exists() and wav_path.stat().st_mtime >= input_path.stat().st_mtime:
            return wav_path

        # Convert using ffmpeg
        cmd = [
            "ffmpeg",
            "-y",
            "-i",
            str(input_path),
            "-ac",
            "1",
            "-ar",
            "16000",
            str(wav_path),
        ]
        print("Converting to WAV:", " ".join(cmd))
        subprocess.run(cmd, check=True, capture_output=True)
        return wav_path
    

## 3) Transcribe one file (test)
+
+Pick the newest file from `audio_inputs/` and transcribe it.
+

In [8]:
import whisper  # type: ignore

    # Pick the newest audio file
exts = {".wav", ".mp3", ".m4a", ".mp4", ".webm", ".ogg", ".flac"}
files = [p for p in AUDIO_DIR.iterdir() if p.is_file() and p.suffix.lower() in exts]
if not files:
        raise RuntimeError("No audio files found. Add an audio file into AI_BACKEND/audio_inputs/ and rerun.")
newest = max(files, key=lambda p: p.stat().st_mtime)
    
wav = ensure_wav(newest)
print("Newest file:", newest.name)
print("Using WAV:", wav.name)
# Choose model size: tiny/base/small/medium/large
model = whisper.load_model("base")
result = model.transcribe(str(wav), language="en")
    
print("\n--- TRANSCRIPT ---\n")
print(result.get("text", "").strip())
    

TypeError: argument of type 'NoneType' is not iterable

## 4) Batch transcribe all audio files
+
+This saves output into `transcripts.jsonl` as one JSON record per file.
+

In [ ]:
def transcribe_all(model_name: str = "base"):
        import whisper  # type: ignore

        model = whisper.load_model(model_name)
        
        # Load already processed files (so we don't duplicate)
        processed = set()
        if TRANSCRIPT_OUT.exists():
            with open(TRANSCRIPT_OUT, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        rec = json.loads(line)
                        processed.add(rec.get("file", ""))
                    except Exception:
                        pass

        exts = {".wav", ".mp3", ".m4a", ".mp4", ".webm", ".ogg", ".flac"}
        files = [p for p in AUDIO_DIR.iterdir() if p.is_file() and p.suffix.lower() in exts]
        files_sorted = sorted(files, key=lambda p: p.stat().st_mtime)
        
        count = 0
        with open(TRANSCRIPT_OUT, "a", encoding="utf-8") as out_f:
            for p in files_sorted:
                if p.name in processed:
                    continue
                wav = ensure_wav(p)
                print("Transcribing:", p.name)
                result = model.transcribe(str(wav), language="en")
                text = (result.get("text", "") or "").strip()
                rec = {
                    "file": p.name,
                    "wav": wav.name,
                    "text": text,
                }
                out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                processed.add(p.name)
                count += 1
        
        print(f"Batch complete. Newly transcribed {count} file(s). Output: {TRANSCRIPT_OUT}")
        
    # Run batch transcribe
    # transcribe_all(model_name="base")
    